## initialization

In [45]:
import pandas as pd
pd.set_option('display.max_columns', None)

In [46]:
from google.colab import files
uploaded = files.upload()

Saving OneDrive_2026-04-21.zip to OneDrive_2026-04-21.zip


## Import

### Extract function

In [47]:
import zipfile
import os
import shutil

def extract_uploaded_data(zip_file_path, extract_to_folder='/content/raw_data'):
    """
    Unzips the uploaded file and ensures the files are not trapped
    inside an unnecessary subfolder (flattens the directory structure).
    """
    print(f"📦 Extracting {zip_file_path}...")

    # Create the target directory if it doesn't exist
    os.makedirs(extract_to_folder, exist_ok=True)

    # 1. Extract the files
    with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
        zip_ref.extractall(extract_to_folder)

    # 2. Check for the "folder inside a folder" problem
    # Get a list of everything directly inside our extraction folder
    contents = os.listdir(extract_to_folder)

    # If there is exactly ONE item, and that item is a directory, we need to flatten it
    if len(contents) == 1:
        nested_folder_path = os.path.join(extract_to_folder, contents[0])

        if os.path.isdir(nested_folder_path):
            print(f"🔧 Found nested folder '{contents[0]}'. Flattening structure...")

            # Move everything from inside the nested folder up to the main folder
            for item in os.listdir(nested_folder_path):
                source = os.path.join(nested_folder_path, item)
                destination = os.path.join(extract_to_folder, item)
                shutil.move(source, destination)

            # Delete the now-empty nested folder
            os.rmdir(nested_folder_path)

    print(f"✅ Extraction complete. Files are ready directly in: {extract_to_folder}")
    return extract_to_folder

### Import read and create df

In [48]:
def year_week_converter(df,day_column='Giorno'):
    # Create unique string year-week ISO (es. '2023-W52', '2024-W01')
    # %G = year ISO, %V = week nr. ISO (01-53)
    df['Anno_Settimana'] = df[day_column].dt.strftime('%G-W%V')

    # move it to 3rd place
    col_settimana = df.pop('Anno_Settimana')
    df.insert(2, 'Anno_Settimana', col_settimana)

    return df

In [49]:
import os
import glob
import pandas as pd

def process_raw_train_data(folder_path):
    """
    Loops through a folder of raw Excel files, extracts 'Elenco Treni'
    (Punctuality and Delays Train List) data, and routes them to two
    separate DataFrames based on direction (Dispari/Odd vs Pari/Even).
    """
    print(f"📂 Scanning folder: {folder_path}")

    # Separate lists for the two directions
    treni_dispari_dfs = []  # Direction 1
    treni_pari_dfs = []     # Direction 2

    all_files = glob.glob(os.path.join(folder_path, '*.xlsx'))

    for file in all_files:
        try:
            # 1. Read metadata.
            meta_df = pd.read_excel(file, header=None, nrows=45, usecols=[0])
            first_cell = str(meta_df.iloc[0, 0]).strip()
            meta_text = ' '.join(meta_df[0].dropna().astype(str))

            # ---------------------------------------------------------
            # ELENCO TRENI (Puntualità e ritardi) ONLY
            # ---------------------------------------------------------
            if first_cell.startswith("Puntualità e ritardi"):
                # Header anchor is 'TRASPORTO' for these specific files
                header_idx = meta_df[meta_df[0].astype(str).str.strip().str.upper() == 'TRASPORTO'].index[0]

                # Extract Even/Odd direction from metadata
                if "Verso = 'Dispari'" in meta_text:
                    direction = 1
                elif "Verso = 'Pari'" in meta_text:
                    direction = 2
                else:
                    direction = None

                # Extract Gruppo/Viste for data consistency
                gruppo_row = meta_df[meta_df[0].astype(str).str.contains("Gruppo =", na=False)]
                if not gruppo_row.empty:
                    gruppo_str = gruppo_row.iloc[0, 0].replace("Gruppo =", "").replace("'", "").strip()
                    short_vista = gruppo_str.split('_')[0]
                else:
                    gruppo_str = None
                    short_vista = None

                df = pd.read_excel(file, skiprows=header_idx)

                df['VISTE'] = gruppo_str
                df['Short_Viste'] = short_vista

                # ---------------------------------------------------------
                # ROUTING LOGIC: Metadata first, fallback to Train Number
                # ---------------------------------------------------------
                if direction is not None:
                    # Apply single direction to whole file
                    df['Direction'] = direction
                    if direction == 1:
                        treni_dispari_dfs.append(df)
                    elif direction == 2:
                        treni_pari_dfs.append(df)
                else:
                    # Fallback to deducing direction row-by-row
                    if 'Treno Commerciale' in df.columns:
                        # Extract only the digits (handles strings like "REG 1234")
                        train_nums = df['Treno Commerciale'].astype(str).str.extract(r'(\d+)')[0]
                        train_nums = pd.to_numeric(train_nums, errors='coerce')

                        # Assign Direction: Odd=1, Even=2
                        df['Direction'] = train_nums.apply(
                            lambda x: 1 if pd.notna(x) and int(x) % 2 != 0
                            else (2 if pd.notna(x) and int(x) % 2 == 0 else None)
                        )

                        # Split DataFrame by direction and append
                        dispari_df = df[df['Direction'] == 1].copy()
                        pari_df = df[df['Direction'] == 2].copy()

                        if not dispari_df.empty:
                            treni_dispari_dfs.append(dispari_df)
                        if not pari_df.empty:
                            treni_pari_dfs.append(pari_df)

                        # Catch rows that couldn't be parsed
                        unparsed_df = df[df['Direction'].isna()]
                        if not unparsed_df.empty:
                            print(f"⚠️ Warning: Could not deduce direction for {len(unparsed_df)} rows in {os.path.basename(file)}.")
                    else:
                        print(f"⚠️ Warning: Direction missing in metadata AND 'Treno Commerciale' not found in {os.path.basename(file)}. Skipping file.")

            # If it's a punctuality or suppression file, it will just quietly skip it now

        except Exception as e:
            print(f"❌ Error reading file {os.path.basename(file)}: {e}")

    # ---------------------------------------------------------
    # FINAL ASSEMBLY & FORMATTING
    # ---------------------------------------------------------

    def format_final_df(df_list):
        df_final = pd.concat(df_list, ignore_index=True) if df_list else pd.DataFrame()

        if not df_final.empty:
            # Reorder specific columns to the front
            short_viste = df_final.pop('Short_Viste')
            df_final.insert(0, 'Short_Viste', short_viste)

            direction_col = df_final.pop('Direction')
            df_final.insert(1, 'Direction', direction_col)

            # Apply the time-formatter if it exists in the global scope
            if 'year_week_converter' in globals():
                df_final = year_week_converter(df_final, day_column='Data')

        return df_final

    # Build the two separate output DataFrames
    df_treni_dispari = format_final_df(treni_dispari_dfs)
    df_treni_pari = format_final_df(treni_pari_dfs)

    print(f"✅ Done! Dispari (Direction 1) rows: {len(df_treni_dispari)} | Pari (Direction 2) rows: {len(df_treni_pari)}")

    # Return exactly two dataframes
    return df_treni_dispari, df_treni_pari

### Clean list

In [50]:
import os

def load_list_from_folder(folder_path):
    """
    Searches a folder for a .txt file.
    - If exactly one is found, reads, cleans, and returns the list.
    - If more than one is found, warns the user and pauses execution.
    - If none are found, prints a warning and returns an empty list so the script continues.
    """
    # 1. Check if the folder even exists first
    if not os.path.exists(folder_path):
        print(f"\n[WARNING] The folder '{folder_path}' does not exist. Skipping.")
        return []

    # Scan the folder for all .txt files
    txt_files = [f for f in os.listdir(folder_path) if f.endswith('.txt')]

    # 2. Handle the file count conditions
    if len(txt_files) == 0:
        # Changed from 'raise' to 'print' and returning an empty list
        print(f"\n[WARNING] No .txt files found in '{folder_path}'. Continuing without data.")
        return []

    elif len(txt_files) > 1:
        print(f"\n[WARNING] Multiple text files found in '{folder_path}':")
        for file in txt_files:
            print(f" - {file}")
        # Pause the execution until the user presses Enter
        input("\nExecution paused. Please resolve the file conflict and press Enter to continue...")
        # Re-scan after the pause in case the user cleaned up the folder
        return load_list_from_folder(folder_path)

    # 3. Exactly one file found -> Get the full path
    target_file = os.path.join(folder_path, txt_files[0])
    print(f"Reading file: {target_file}")

    # 4. Clean and parse the file (your original logic)
    with open(target_file, 'r', encoding='utf-8') as file:
        lines = file.readlines()

    clean_list = [line.strip().replace('**', '') for line in lines if line.strip()]

    if len(clean_list) > 1:
        clean_list = clean_list[1:]

    return clean_list

### Fix dates and times and weeknumber

In [51]:
def clean_treni_timestamps(df):
    """
    Combines Date and Time columns into full datetime objects.
    Case-insensitive and space-insensitive column matching.
    """
    df = df.copy()

    # 1. Define your pairs in purely lowercase
    pairs = [
        ('data origine reale', 'ora origine reale'),
        ('data destino reale', 'ora destino reale'),
        ('data entrata giurisdizione', 'ora entrata giurisdizione'),
        ('data uscita giurisdizione', 'ora uscita giurisdizione')
    ]

    # 2. Create a translation dictionary: { 'lowercase_clean_name': 'Actual Original Name' }
    # Example: 'data origine reale' -> 'Data Origine Reale '
    col_map = {str(col).lower().strip(): col for col in df.columns}

    for date_target, time_target in pairs:

        # 3. Check if our targets exist in the lowercase map
        if date_target in col_map and time_target in col_map:

            # 4. Fetch the exact original column names
            actual_date_col = col_map[date_target]
            actual_time_col = col_map[time_target]

            # 5. Do the combination using the real column names
            df[actual_time_col] = pd.to_datetime(
                df[actual_date_col].dt.strftime('%Y-%m-%d') + ' ' + df[actual_time_col].dt.strftime('%H:%M:%S'),
                errors='coerce'
            )

    return df

In [52]:
import pandas as pd
import numpy as np

def run_sanity_checks(df):
    """
    Enforces chronological integrity: Origin <= Entry <= Exit <= Destination.
    If a train violates this logic at any transit point, its entire
    trajectory for that day is dropped.

    Returns:
        df_clean (DataFrame): The sanitized dataset.
        diagnostic_log (DataFrame): A log of the specific trains, stations, and reasons for failure.
    """
    df_clean = df.copy()

    # 1. Define the chronological conditions
    cond1 = df_clean['Ora origine reale'] > df_clean['Ora Entrata Giurisdizione']
    cond2 = df_clean['Ora Entrata Giurisdizione'] > df_clean['Ora Uscita Giurisdizione']
    cond3 = df_clean['Ora Uscita Giurisdizione'] > df_clean['Ora destino reale']

    # 2. Isolate the rows that failed any of the checks
    invalid_mask = cond1 | cond2 | cond3
    invalid_rows = df_clean[invalid_mask].copy()

    if not invalid_rows.empty:
        # 3. Tag the exact failure reason using np.select
        conditions = [
            invalid_rows['Ora origine reale'] > invalid_rows['Ora Entrata Giurisdizione'],
            invalid_rows['Ora Entrata Giurisdizione'] > invalid_rows['Ora Uscita Giurisdizione'],
            invalid_rows['Ora Uscita Giurisdizione'] > invalid_rows['Ora destino reale']
        ]

        choices = [
            'Failure: Origin > Entry',
            'Failure: Entry > Exit',
            'Failure: Exit > Destination'
        ]

        invalid_rows['Failure_Reason'] = np.select(conditions, choices, default='Multiple/Unknown')

        # 4. Create the diagnostic log with the requested columns
        diagnostic_log = invalid_rows[['Data Origine Reale', 'Treno Commerciale', 'Località passaggio', 'Failure_Reason']].copy()

        # 5. Get the unique Date & Train combinations that must be purged entirely
        invalid_trains = invalid_rows[['Data Origine Reale', 'Treno Commerciale']].drop_duplicates()

        # 6. Filter out any train that exists in the invalid list
        df_merged = df_clean.merge(invalid_trains, on=['Data Origine Reale', 'Treno Commerciale'], how='left', indicator=True)
        df_clean = df_merged[df_merged['_merge'] == 'left_only'].drop(columns=['_merge'])

        print(f"⚠️ Sanity Check: Dropped {len(invalid_trains)} invalid train journeys.")
        print("🔍 Sample of failed transits:")
        print(diagnostic_log.head())

    else:
        diagnostic_log = pd.DataFrame()
        print("✅ Sanity Check: All timelines are chronologically valid.")

    return diagnostic_log

In [53]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

def plot_delay_distribution_sns(even_df, odd_df, delay_col='Ritardo Entrata Territorio'):
    """
    Plots the frequency distribution of delays for two datasets using Seaborn.
    """
    # Create a temporary combined DataFrame for easy Seaborn plotting
    even_temp = pd.DataFrame({delay_col: even_df[delay_col].dropna(), 'Dataset': 'Even'})
    odd_temp = pd.DataFrame({delay_col: odd_df[delay_col].dropna(), 'Dataset': 'Odd'})
    combined_df = pd.concat([even_temp, odd_temp])

    plt.figure(figsize=(12, 6))

    # Plot using step element to avoid muddy overlapping colors
    sns.histplot(
        data=combined_df,
        x=delay_col,
        hue='Dataset',
        bins=100,
        element='step', # 'step' outlines are clearer than overlapping filled bars
        palette=['steelblue', 'darkorange'],
        alpha=0.6
    )

    plt.title(f'Arrival Delay Distribution: Even vs. Odd', fontsize=14)
    plt.xlabel('Delay (minutes)', fontsize=12)
    plt.ylabel('Frequency (Log Scale)', fontsize=12)
    plt.yscale('log')
    plt.grid(axis='y', alpha=0.3)

    # Seaborn adds a legend automatically when using 'hue'
    plt.show()

## Stops filtering and things like that

### Remove columns and split df

In [54]:
import pandas as pd

def clean_df(df, columns_to_drop=None):
    """
    Removes unnecessary columns safely, then splits the main dataset
    into Dispari (Forward) and Pari (Reverse) dataframes.
    """
    # 1. Drop junk columns (if a list is provided)
    if columns_to_drop:
        # errors='ignore' ensures the code won't crash if a column
        # is already missing or spelled differently.
        df = df.drop(columns=columns_to_drop, errors='ignore')
        col = df.pop('Treno Commerciale')
        df.insert(3, 'Treno Commerciale', col)
        col = df.pop('Trasporto')
        df.insert(len(df.columns), 'Trasporto', col)
    return df

### Filter the stops, and reorder

In [55]:
import pandas as pd

def sort_treni_geographically(df, master_list=None):
    """
    Filters and sorts train data geographically, and adds geographic order
    and stop sequence columns.
    """
    df = df.copy()

    # --- STEP 1: Determine the station order ---
    if master_list:
        direction = df['Direction'].mode()[0] if 'Direction' in df.columns else 1
        working_list = master_list[::-1] if direction == 2 else master_list
    else:
        longest_train = df.groupby('Treno Commerciale')['Località passaggio'].nunique().idxmax()
        train_route = df[df['Treno Commerciale'] == longest_train].sort_values('Ora Uscita Giurisdizione')
        working_list = train_route['Località passaggio'].drop_duplicates().tolist()

    # --- STEP 2: Filter out excess stations ---
    df = df[df['Località passaggio'].isin(working_list)]

    # --- STEP 3: Lock in the geographical order ---
    df['Località passaggio'] = pd.Categorical(
        df['Località passaggio'],
        categories=working_list,
        ordered=True
    )

    # --- STEP 4: Sort chronologically and geographically ---
    sort_cols = ['Data Origine Reale', 'Ora origine reale', 'Treno Commerciale', 'Località passaggio']
    sort_asc = [True, True, False, True]

    df = df.sort_values(by=sort_cols, ascending=sort_asc).reset_index(drop=True)

    # --- STEP 5: Generate Map Order (Geo_Order) ---
    # SIMPLIFIED: Since the column is already a Categorical, just pull its underlying number
    df['Geo_Order'] = df['Località passaggio'].cat.codes + 1

    # --- STEP 6: Generate Stop Sequence ---
    df['Stop_Sequence'] = df.groupby(['Data Origine Reale', 'Treno Commerciale']).cumcount() + 1

    return df, working_list

In [56]:
import pandas as pd

def anonymize_stations(df, station_col='Località passaggio'):
    """
    Creates an anonymized, geographically ordered node column to protect
    proprietary infrastructure data in the final thesis plots.
    """
    df = df.copy()

    # Extract the ordered list of real stations
    ordered_stations = df[station_col].cat.categories

    # Create a mapping dictionary: {'Milano Centrale': 'Node_01', 'Milano Lambrate': 'Node_02', ...}
    anon_mapping = {station: f"Location_{i+1:02d}" for i, station in enumerate(ordered_stations)}

    # Apply the mapping to create a new column
    df['Node_Anon'] = df[station_col].map(anon_mapping)

    # Lock the new column as an ordered Categorical so the plots respect the geography
    df['Node_Anon'] = pd.Categorical(
        df['Node_Anon'],
        categories=list(anon_mapping.values()),
        ordered=True
    )

    return df, anon_mapping

### Calcolo delle metrics

In [57]:
import pandas as pd
import numpy as np

def calculate_macro_weekly_summary(df, dest_delay_col='Ritardo destino', stop_delay_col='Ritardo Uscita Territorio', punct_threshold=5.5):
    """
    Calculates the Macro Weekly Summary for the entire route.
    Includes overall core indicators and Directional Asymmetry (Diff Punct).
    """
    print(f"📊 Calculating Macro Weekly (Threshold <= {punct_threshold} mins)...")

    # -----0. concatenate------
    df_out = df.copy()

    # --- 1. OVERALL ALL-STOPS METRICS ---
    # Every row is a stop, so we aggregate natively
    all_stops_overall = df_out.groupby('Anno_Settimana').agg(
        All_Stops_Avg_Delay=(stop_delay_col, 'mean'),
        All_Stops_Punctuality=(stop_delay_col, lambda x: (x <= punct_threshold).mean() * 100)
    ).reset_index()

    # --- 2. OVERALL DESTINATION METRICS ---
    # Drop duplicates to isolate exactly ONE row per individual train run
    print("   🔍 Isolating individual train runs...")
    df_dest = df_out.drop_duplicates(subset=['Data', 'Trasporto']).copy()

    dest_overall = df_dest.groupby('Anno_Settimana').agg(
        Dest_Avg_Delay=(dest_delay_col, 'mean'),
        Dest_Punctuality=(dest_delay_col, lambda x: (x <= punct_threshold).mean() * 100),
        Treni_Circolati=('Trasporto', 'count') # FIXED: Now it counts the actual total runs, not unique IDs
    ).reset_index()

    # Merge overall metrics
    df_overall = pd.merge(all_stops_overall, dest_overall, on='Anno_Settimana', how='inner')

    # --- 3. DIRECTIONAL ASYMMETRY (Diff Punct & Volumes) ---
    print("   ⚖️ Calculating directional asymmetry...")
    # Group by Week AND Direction
    dir_stats = df_dest.groupby(['Anno_Settimana', 'Direction']).agg(
        Punct=(dest_delay_col, lambda x: (x <= punct_threshold).mean() * 100),
        Circ=('Trasporto', 'count') # FIXED: Counts runs per direction
    ).reset_index()

    # Pivot so directions become columns (1=Dispari, 2=Pari)
    df_pivot = dir_stats.pivot(index='Anno_Settimana', columns='Direction', values=['Punct', 'Circ'])
    # Flatten multi-level columns
    df_pivot.columns = [f"{col[0]}_{int(col[1])}" if pd.notna(col[1]) else col[0] for col in df_pivot.columns]
    df_pivot = df_pivot.reset_index()

    # Calculate Differences using .get() to safely handle weeks that might miss a direction
    df_pivot['Diff_True_Punct'] = df_pivot.get('Punct_1', np.nan) - df_pivot.get('Punct_2', np.nan)

    c1 = df_pivot.get('Circ_1', np.nan)
    c2 = df_pivot.get('Circ_2', np.nan)
    df_pivot['Diff_Circolati_Norm'] = ((c1 - c2) / (c1 + c2).replace(0, np.nan)) * 100

    df_asym = df_pivot[['Anno_Settimana', 'Diff_True_Punct', 'Diff_Circolati_Norm']]

    # --- 4. FINAL ASSEMBLY ---
    master_summary = pd.merge(df_overall, df_asym, on='Anno_Settimana', how='left')

    # Insert Route Name if you want to keep the identifier
    if 'Short_Viste' in df_out.columns:
        master_summary.insert(0, 'Short_Viste', df_out['Short_Viste'].iloc[0])

    print(f"✅ Summary Complete. Generated {len(master_summary)} weeks of macro data.")
    return master_summary.round(2)

### Calcolo KPIs

In [58]:
import numpy as np

def calculate_historical_trends(df, metrics_config, window=4):
    """
    Calculates historical rolling averages, deltas, alert statuses, and target thresholds.
    Requires exactly 'window' previous weeks to calculate an average.
    """
    print(f"📈 Calculating {window}-week historical trends & thresholds...")
    df_out = df.copy()

    # Sort chronologically for accurate rolling math
    if 'Short_Viste' in df_out.columns:
        df_out = df_out.sort_values(by=['Short_Viste', 'Anno_Settimana']).reset_index(drop=True)
        grouped = df_out.groupby('Short_Viste')
    else:
        # If no route name is provided, just group the whole dataframe as one
        df_out = df_out.sort_values(by=['Anno_Settimana']).reset_index(drop=True)
        df_out['Dummy_Group'] = 1
        grouped = df_out.groupby('Dummy_Group')

    for metric, (alert_direction, threshold) in metrics_config.items():
        if metric not in df_out.columns:
            continue

        # 1. STRICT WINDOWING: Shift(1) ensures the current week is excluded from its own past average
        hist_avg = grouped[metric].transform(lambda x: x.rolling(window=window).mean().shift(1))

        ma_col = f'RollingAvg_{window}w_{metric}'
        delta_col = f'Delta_{window}w_{metric}'
        status_col = f'Status_{window}w_{metric}'

        df_out[ma_col] = hist_avg
        df_out[delta_col] = df_out[metric] - hist_avg

        # 2. SAFE THRESHOLDS & TARGET CALCULATIONS
        if alert_direction == 'higher_is_bad':
            # Adds a single 'Target' column (Maximum allowable limit)
            df_out[f'Target_{window}w_{metric}'] = df_out[ma_col] + threshold
            df_out[status_col] = np.where(
                df_out[delta_col].isna(),
                np.nan,
                np.where(df_out[delta_col] > threshold, 1, 0)
            )

        elif alert_direction == 'lower_is_bad':
            # Adds a single 'Target' column (Minimum allowable limit)
            # Using abs() ensures we subtract the threshold correctly even if configured as a negative number
            df_out[f'Target_{window}w_{metric}'] = df_out[ma_col] - abs(threshold)
            df_out[status_col] = np.where(
                df_out[delta_col].isna(),
                np.nan,
                np.where(df_out[delta_col] < -abs(threshold), 1, 0)
            )

        elif alert_direction == 'either_is_bad':
            # Adds two columns: 'TargetUpper' and 'TargetLower'
            df_out[f'TargetUpper_{window}w_{metric}'] = df_out[ma_col] + abs(threshold)
            df_out[f'TargetLower_{window}w_{metric}'] = df_out[ma_col] - abs(threshold)

            df_out[status_col] = np.where(
                df_out[delta_col].isna(),
                np.nan,
                np.where(abs(df_out[delta_col]) > abs(threshold), 1, 0)
            )

    if 'Dummy_Group' in df_out.columns:
        df_out.drop(columns=['Dummy_Group'], inplace=True)

    print("✅ Historical trends complete.")
    return df_out.round(2)

## Wrapper Functions

### Wrapper 1

In [59]:
def import_and_process_data(zip_path, extract_to_folder='/content/raw_data'):

  # unzipping the zip file
  folder_path =extract_uploaded_data(zip_file_path=zip_path, extract_to_folder=extract_to_folder)

  # reading and categorizing the different files
  df_odd,df_even=process_raw_train_data(folder_path)

  # find stop list
  lista=load_list_from_folder(folder_path)

  return df_odd,df_even,lista

### Wrapper 2

In [60]:
def data_manipulation(df, columns_to_drop=None, station_list=None):

  # Clean the dataframes
  df_1 = clean_treni_timestamps(df=df)

  # Run sanity checks
  run_sanity_checks(df_1)

  # Remove useless columns
  df_2 = clean_df(df=df_1, columns_to_drop=columns_to_drop)

  # Sort stops in order
  df_3, working_list = sort_treni_geographically(df=df_2, master_list=station_list)

  return df_3, working_list

### Wrapper 3

In [61]:
def micro_df(dis, par, list):
    # 1. Combine your two geographically sorted dataframes back into one
    df_powerbi = pd.concat([dis, par], ignore_index=True)

    # Ensure 'Località passaggio' is categorical before anonymization
    df_powerbi['Località passaggio'] = pd.Categorical(
        df_powerbi['Località passaggio'],
        categories=list,
        ordered=True
    )

    # 1.5 add anonymization
    df_powerbi, station_anonymization_mapping = anonymize_stations(df_powerbi)

    # 2. Add Readable Direction Names for the Slicers
    df_powerbi['Direction_Name'] = df_powerbi['Direction'].map({1: 'Dispari (Forward)', 2: 'Pari (Reverse)'})

    # 3. Translate the column names to be clean
    # rename_dict = {
    #     'Treno Commerciale': 'Train_ID',
    #     'Data': 'Date',
    #     'Località passaggio': 'Station',
    #     'Ritardo Uscita Territorio': 'Station_Delay_Min',
    #     'Ritardo destino': 'Final_Dest_Delay_Min'
    # }
    # df_powerbi = df_powerbi.rename(columns=rename_dict)

    # 6. Export Table 1
    df_powerbi.to_csv('1_Granular_Train_Data.csv', index=False, sep=';', decimal=',', encoding='utf-8-sig')
    print("✅ Table 1 (Granular Data) Exported.")

    return df_powerbi, station_anonymization_mapping

### Wrapper 4

In [62]:
def macro_df(micro_df, metrics_config, window=4, dest_delay_col='Ritardo destino', stop_delay_col='Ritardo Uscita Territorio', punct_threshold=5.5):
    # 1. Use your existing function on df_3 to get the base weekly stats
    summary_df = calculate_macro_weekly_summary(df=micro_df, dest_delay_col=dest_delay_col, stop_delay_col=stop_delay_col, punct_threshold=punct_threshold)

    # 2. Use your existing function to calculate the 4-week rolling math
    final_macro_df = calculate_historical_trends(summary_df, metrics_config)

    # In your final export cell, add this logic for Table 2:
    print("🔢 Adding Sort Keys to Macro Table...")

    # Create a numerical key: 2026-W01 becomes 202601
    # We strip the '-W' and convert to integer
    final_macro_df['Week_Sort_Key'] = (
        final_macro_df['Anno_Settimana']
        .str.replace('-W', '', regex=False)
        .astype(int)
    )

    # Now export again
    final_macro_df.to_csv('2_Macro_Weekly_Summary.csv', index=False, sep=';', decimal=',', encoding='utf-8-sig')
    print("✅ Macro Table with native Sort Key exported.")

    return final_macro_df

## Variables Definition

In [63]:
### variables for wrapper 2
# 1. Define the unnecessary columns you want to throw away
junk_cols = [
    'Tipo Percorso Traccia Progr.',
    'Tipo treno',
    'Segm Merc. Puntualità',
    'Categoria',
    'Categoria commerciale',
    'Motivo Esclusione',
    'T Rit (%OS)'
]

### variables for wrapper 4
# Define your alert rules: { 'Column Name': ('alert_direction', threshold_value) }
macro_trend_config = {
    'Dest_Avg_Delay':        ('higher_is_bad', 2.0),
    'Dest_Punctuality':      ('lower_is_bad', 6.0),
    'All_Stops_Avg_Delay':   ('higher_is_bad', 2.0),
    'All_Stops_Punctuality': ('lower_is_bad', 6.0),
    'Diff_True_Punct':       ('either_is_bad', 2.0),
    'Diff_Circolati_Norm':   ('either_is_bad', 8.0)
}


## SCRIPT OPERATIVO
Modificare solo questo durante l'utilizzo quotidiano

In [68]:
import warnings

# =====================================================================
# PIPELINE SETUP
# =====================================================================
# Mute the harmless but annoying openpyxl style warnings
warnings.filterwarnings('ignore', category=UserWarning, module='openpyxl')

print("\n" + "="*55)
print("🚀 STARTING DATA PIPELINE")
print("="*55 + "\n")


# =====================================================================
# WRAPPER 1: IMPORT & PROCESS
# =====================================================================
# Qui va cambiata solo la prima variabile 'zip_path'
print("▶️ STEP 1: Extracting and Importing Data...")
odd, even, lista = import_and_process_data(zip_path='/content/OneDrive_2026-04-21.zip', extract_to_folder='/content/raw_data2')


# =====================================================================
# WRAPPER 2: DATA MANIPULATION
# =====================================================================
# NON TOCCARE
print("\n▶️ STEP 2: Cleaning and Manipulating Directional Data...")
odd_clean, odd_list = data_manipulation(odd, columns_to_drop=junk_cols, station_list=lista)
even_clean, even_list = data_manipulation(even, columns_to_drop=junk_cols, station_list=lista)


# =====================================================================
# WRAPPER 3: MICRO TABLE
# =====================================================================
# NON TOCCARE
print("\n▶️ STEP 3: Generating Micro Table (Map & Daily Matrix)...")
the_micro_df, anonymization_mapping = micro_df(odd_clean, even_clean, odd_list)


# =====================================================================
# WRAPPER 4: MACRO TABLE
# =====================================================================
# NON TOCCARE
print("\n▶️ STEP 4: Generating Macro Summary Table (Trends & Alerts)...")
the_macro_df = macro_df(micro_df=the_micro_df, metrics_config=macro_trend_config, window=4)

print("\n" + "="*55)
print("✨ PIPELINE COMPLETE!")
print("="*55 + "\n")


🚀 STARTING DATA PIPELINE

▶️ STEP 1: Extracting and Importing Data...
📦 Extracting /content/OneDrive_2026-04-21.zip...
🔧 Found nested folder 'DATI SINGOLA LINEA MILANO MORTARA'. Flattening structure...
✅ Extraction complete. Files are ready directly in: /content/raw_data2
📂 Scanning folder: /content/raw_data2
✅ Done! Dispari (Direction 1) rows: 31308 | Pari (Direction 2) rows: 30927
Reading file: /content/raw_data2/Ordine Località.txt

▶️ STEP 2: Cleaning and Manipulating Directional Data...
⚠️ Sanity Check: Dropped 43 invalid train journeys.
🔍 Sample of failed transits:
     Data Origine Reale  Treno Commerciale Località passaggio  \
382          2026-01-02              10045    Cippo0.00/88.52   
467          2026-01-02              10065    Cippo0.00/88.52   
710          2026-01-03              10045    Cippo0.00/88.52   
1349         2026-01-05              10065    Cippo0.00/88.52   
2427         2026-01-08              10073    Cippo0.00/88.52   

                   Failure_Rea

In [67]:
the_micro_df

,Short_Viste,Direction,Anno_Settimana,Treno Commerciale,Data,Località Origine Reale,Ritardo origine,Data Origine Reale,Ora origine reale,Località Destino Reale,Ritardo destino,Data destino reale,Ora destino reale,Località Passaggio,Ritardo Entrata Territorio,Data Entrata Giurisdizione,Ora Entrata Giurisdizione,Località passaggio,Ritardo Uscita Territorio,Data Uscita Giurisdizione,Ora Uscita Giurisdizione,Cliente,VISTE,Trasporto,Geo_Order,Stop_Sequence,Node_Anon,Direction_Name
0,16. MI1,1,2026-W24,10955,2026-06-14,MI.P.GARIBALDI,0.5,2026-06-14,2026-06-14 06:17:30,Stradella,1.0,2026-06-14,2026-06-14 07:54:00,MI.P.GARIBALDI,0.0,NaT,NaT,MI.P.GARIBALDI,0.5,2026-06-14,2026-06-14 06:17:30,063-TRENORD-BP,16. MI1_MILANO PG-STRADELLA,110955,1,1,Location_01,Dispari (Forward)
1,16. MI1,1,2026-W24,10955,2026-06-14,MI.P.GARIBALDI,0.5,2026-06-14,2026-06-14 06:17:30,Stradella,1.0,2026-06-14,2026-06-14 07:54:00,PM Turro,-0.5,2026-06-14,2026-06-14 06:23:30,PM Turro,-0.5,2026-06-14,2026-06-14 06:23:30,063-TRENORD-BP,16. MI1_MILANO PG-STRADELLA,110955,6,2,Location_06,Dispari (Forward)
2,16. MI1,1,2026-W24,10955,2026-06-14,MI.P.GARIBALDI,0.5,2026-06-14,2026-06-14 06:17:30,Stradella,1.0,2026-06-14,2026-06-14 07:54:00,MILANO LAMBRATE,1.5,2026-06-14,2026-06-14 06:30:30,MILANO LAMBRATE,2.5,2026-06-14,2026-06-14 06:32:30,063-TRENORD-BP,16. MI1_MILANO PG-STRADELLA,110955,7,3,Location_07,Dispari (Forward)
3,16. MI1,1,2026-W24,10955,2026-06-14,MI.P.GARIBALDI,0.5,2026-06-14,2026-06-14 06:17:30,Stradella,1.0,2026-06-14,2026-06-14 07:54:00,MILANO ROGOREDO,-2.5,2026-06-14,2026-06-14 06:39:30,MILANO ROGOREDO,0.5,2026-06-14,2026-06-14 06:43:30,063-TRENORD-BP,16. MI1_MILANO PG-STRADELLA,110955,8,4,Location_08,Dispari (Forward)
4,16. MI1,1,2026-W24,10955,2026-06-14,MI.P.GARIBALDI,0.5,2026-06-14,2026-06-14 06:17:30,Stradella,1.0,2026-06-14,2026-06-14 07:54:00,Locate Triulzi,-0.5,2026-06-14,2026-06-14 06:48:30,Locate Triulzi,0.0,2026-06-14,2026-06-14 06:49:00,063-TRENORD-BP,16. MI1_MILANO PG-STRADELLA,110955,9,5,Location_09,Dispari (Forward)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11497,16. MI1,2,2026-W27,10480,2026-07-02,PIACENZA,0.0,2026-07-02,2026-07-02 19:35:00,MI.P.GARIBALDI,9.5,2026-07-02,2026-07-02 21:45:30,PM Turro,14.5,2026-07-02,2026-07-02 21:35:30,PM Turro,14.5,2026-07-02,2026-07-02 21:35:30,063-TRENORD-BP,16. MI1_MILANO PG-STRADELLA,110480,23,23,Location_06,Pari (Reverse)
11498,16. MI1,2,2026-W27,10480,2026-07-02,PIACENZA,0.0,2026-07-02,2026-07-02 19:35:00,MI.P.GARIBALDI,9.5,2026-07-02,2026-07-02 21:45:30,Tr. B./PC Seveso,14.0,2026-07-02,2026-07-02 21:37:00,Tr. B./PC Seveso,14.0,2026-07-02,2026-07-02 21:37:00,063-TRENORD-BP,16. MI1_MILANO PG-STRADELLA,110480,24,24,Location_05,Pari (Reverse)
11499,16. MI1,2,2026-W27,10480,2026-07-02,PIACENZA,0.0,2026-07-02,2026-07-02 19:35:00,MI.P.GARIBALDI,9.5,2026-07-02,2026-07-02 21:45:30,Biv. Musocco,14.0,2026-07-02,2026-07-02 21:40:00,Biv. Musocco,14.0,2026-07-02,2026-07-02 21:40:00,063-TRENORD-BP,16. MI1_MILANO PG-STRADELLA,110480,25,25,Location_04,Pari (Reverse)
11500,16. MI1,2,2026-W27,10480,2026-07-02,PIACENZA,0.0,2026-07-02,2026-07-02 19:35:00,MI.P.GARIBALDI,9.5,2026-07-02,2026-07-02 21:45:30,P.M. GHISOLFA,10.5,2026-07-02,2026-07-02 21:41:30,P.M. GHISOLFA,10.5,2026-07-02,2026-07-02 21:41:30,063-TRENORD-BP,16. MI1_MILANO PG-STRADELLA,110480,26,26,Location_02,Pari (Reverse)
